# 00 — Data Verification
Confirms the sorted SIPaKMeD dataset is correct, tests the slide-ID
parser against every real filename, and quantifies leakage risk
(how many cells share a source slide) — this last part gives us a
concrete number for the paper's motivation section.

In [1]:
import os
import re
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
from PIL import Image

DATASET_ROOT = 'dataset'
CLASS_NAMES = [
    'Dyskeratotic', 'Koilocytotic', 'Metaplastic',
    'Parabasal', 'Superficial-Intermediate',
]

## 1. Confirm counts per class match expected SIPaKMeD totals

In [6]:
import os
print("Current directory:", os.getcwd())
print("Contents:", os.listdir('.'))

Current directory: C:\developer\PapSmear_AI_v2
Contents: ['.git', 'dataset', 'notebooks', 'outputs', 'raw', 'README.md', 'requirements.txt', 'sort_dataset.py', 'src', 'venv']


In [7]:
expected = {
    'Dyskeratotic': 813, 'Koilocytotic': 825, 'Metaplastic': 793,
    'Parabasal': 787, 'Superficial-Intermediate': 831,
}

all_files = {}  # class_name -> list of filenames
for class_name in CLASS_NAMES:
    class_path = os.path.join(DATASET_ROOT, class_name)
    files = sorted(f for f in os.listdir(class_path) if f.lower().endswith('.bmp'))
    all_files[class_name] = files
    status = "✅" if len(files) == expected[class_name] else "⚠️ MISMATCH"
    print(f"{class_name}: {len(files)} (expected {expected[class_name]}) {status}")

total = sum(len(v) for v in all_files.values())
print(f"\nTotal: {total} (expected 4049)")

Dyskeratotic: 813 (expected 813) ✅
Koilocytotic: 825 (expected 825) ✅
Metaplastic: 793 (expected 793) ✅
Parabasal: 787 (expected 787) ✅
Superficial-Intermediate: 831 (expected 831) ✅

Total: 4049 (expected 4049)


## 2. Visually confirm one sample image per class
If any image looks wrong (blank, corrupted, wrong color channels),
stop here before anything else depends on it.

In [8]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, class_name in zip(axes, CLASS_NAMES):
    sample_file = all_files[class_name][0]
    img_path = os.path.join(DATASET_ROOT, class_name, sample_file)
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(f"{class_name}\n{sample_file}\n{img.size}")
    ax.axis('off')
plt.tight_layout()
plt.savefig('outputs/figures/00_sample_images_per_class.png', dpi=150)
plt.show()

<Figure size 2000x400 with 5 Axes>

## 3. Test the slide-ID parser against EVERY real filename
Pattern confirmed from the actual download: {source_id}_{crop_number}.bmp
e.g. "001_04.bmp" -> source slide "001", crop 4

This must succeed on all 4,049 filenames before we trust it for the
actual CV split — a single unparsed filename would silently drop
that image from grouping and break the leakage-safe guarantee.

In [9]:
SLIDE_ID_PATTERN = re.compile(r'^(\d+)_(\d+)\.bmp$', re.IGNORECASE)

def parse_slide_id(filename):
    match = SLIDE_ID_PATTERN.match(filename)
    if match is None:
        return None
    return match.group(1)  # the source slide ID, e.g. "001"

parse_failures = []
records = []  # (class_name, filename, slide_id)

for class_name in CLASS_NAMES:
    for fname in all_files[class_name]:
        slide_id = parse_slide_id(fname)
        if slide_id is None:
            parse_failures.append((class_name, fname))
        else:
            records.append((class_name, fname, slide_id))

print(f"Successfully parsed: {len(records)} / {total}")
if parse_failures:
    print(f"\n⚠️  {len(parse_failures)} FILENAMES FAILED TO PARSE — must fix before CV split:")
    for cls, fname in parse_failures[:20]:
        print(f"  {cls}/{fname}")
else:
    print("✅ All filenames parsed successfully.")

Successfully parsed: 4049 / 4049
✅ All filenames parsed successfully.


## 4. Quantify leakage risk — how many cells share a source slide?
This is the actual evidence for the paper's novelty claim: if most
slides only contribute 1 cell, leakage risk is minor. If most slides
contribute several, naive random splitting has real leakage exposure.

IMPORTANT: slide IDs are only unique *within* a class folder (e.g.
Dyskeratotic's "001" and Koilocytotic's "001" are different source
slides, not the same one) — so grouping must be keyed on
(class_name, slide_id) together, not slide_id alone.

In [10]:
slide_cell_counts = Counter()
for class_name, fname, slide_id in records:
    slide_cell_counts[(class_name, slide_id)] += 1

counts_only = list(slide_cell_counts.values())
n_slides = len(slide_cell_counts)
n_single_cell_slides = sum(1 for c in counts_only if c == 1)
n_multi_cell_slides = n_slides - n_single_cell_slides

print(f"Total unique source slides: {n_slides}")
print(f"Slides contributing exactly 1 cell: {n_single_cell_slides} ({n_single_cell_slides/n_slides*100:.1f}%)")
print(f"Slides contributing 2+ cells: {n_multi_cell_slides} ({n_multi_cell_slides/n_slides*100:.1f}%)")
print(f"Max cells from a single slide: {max(counts_only)}")
print(f"Mean cells per slide: {total/n_slides:.2f}")

# Distribution table
dist = Counter(counts_only)
print("\nCells-per-slide distribution:")
for n_cells in sorted(dist.keys()):
    print(f"  {n_cells} cell(s): {dist[n_cells]} slides")

Total unique source slides: 966
Slides contributing exactly 1 cell: 244 (25.3%)
Slides contributing 2+ cells: 722 (74.7%)
Max cells from a single slide: 26
Mean cells per slide: 4.19

Cells-per-slide distribution:
  1 cell(s): 244 slides
  2 cell(s): 189 slides
  3 cell(s): 109 slides
  4 cell(s): 97 slides
  5 cell(s): 77 slides
  6 cell(s): 47 slides
  7 cell(s): 51 slides
  8 cell(s): 36 slides
  9 cell(s): 26 slides
  10 cell(s): 22 slides
  11 cell(s): 18 slides
  12 cell(s): 9 slides
  13 cell(s): 11 slides
  14 cell(s): 9 slides
  15 cell(s): 8 slides
  16 cell(s): 5 slides
  17 cell(s): 2 slides
  19 cell(s): 1 slides
  21 cell(s): 1 slides
  22 cell(s): 1 slides
  24 cell(s): 1 slides
  25 cell(s): 1 slides
  26 cell(s): 1 slides


## 5. Per-class breakdown of leakage exposure
Useful for the paper's methods section — shows this isn't uniform
across classes.

In [11]:
per_class_multi = defaultdict(lambda: [0, 0])  # class -> [single_cell_slides, multi_cell_slides]
for (class_name, slide_id), count in slide_cell_counts.items():
    if count == 1:
        per_class_multi[class_name][0] += 1
    else:
        per_class_multi[class_name][1] += 1

print(f"{'Class':<28} {'Single-cell slides':<20} {'Multi-cell slides':<20}")
for class_name in CLASS_NAMES:
    single, multi = per_class_multi[class_name]
    print(f"{class_name:<28} {single:<20} {multi:<20}")

Class                        Single-cell slides   Multi-cell slides   
Dyskeratotic                 62                   161                 
Koilocytotic                 80                   158                 
Metaplastic                  93                   178                 
Parabasal                    6                    102                 
Superficial-Intermediate     3                    123                 


## Summary
If parsing succeeded on all 4,049 files and the multi-cell-slide
percentage is non-trivial (expect well over half, based on 4,049
cells from 966 slides ≈ 4.2 cells/slide on average), this confirms
the leakage-risk motivation is real and gives concrete numbers to
cite in the paper's methodology section — not just the general
claim from the SIPaKMeD dataset description.